# STAR Demo

## Part 1: Legacy API

Using `LiquidHandler` + `STARChatterboxBackend` to demonstrate PIP, Head96, iSWAP, and CoRe gripper.

In [1]:
%load_ext autoreload
%autoreload 2

### Setup

In [ ]:
from pylabrobot.legacy.liquid_handling import LiquidHandler
from pylabrobot.legacy.liquid_handling.backends.hamilton.STAR_chatterbox import (
  STARChatterboxBackend,
)
from pylabrobot.resources import (
  PLT_CAR_L5AC_A00,
  TIP_CAR_480_A00,
  Cor_96_wellplate_360ul_Fb,
  hamilton_96_tiprack_1000uL_filter,
)
from pylabrobot.resources.hamilton import STARDeck

backend = STARChatterboxBackend()
deck = STARDeck()
lh = LiquidHandler(backend=backend, deck=deck)

In [3]:
# Deck layout
tip_car = TIP_CAR_480_A00(name="tip_carrier")
tip_car[0] = hamilton_96_tiprack_1000uL_filter(name="tips_01")
tip_car[1] = hamilton_96_tiprack_1000uL_filter(name="tips_02")
deck.assign_child_resource(tip_car, rails=3)

plt_car = PLT_CAR_L5AC_A00(name="plate_carrier")
plt_car[0] = Cor_96_wellplate_360ul_Fb(name="plate_01")
plt_car[1] = Cor_96_wellplate_360ul_Fb(name="plate_02")
deck.assign_child_resource(plt_car, rails=15)

In [4]:
await lh.setup()

### PIP: independent channel pipetting

In [5]:
tiprack = deck.get_resource("tips_01")
plate = deck.get_resource("plate_01")

# Pick up tips on channels 0-2
await lh.pick_up_tips(tiprack["A1:C1"])

# Aspirate from wells A1-C1
await lh.aspirate(plate["A1:C1"], vols=[100.0, 50.0, 200.0])

# Dispense into wells D1-F1
await lh.dispense(plate["D1:F1"], vols=[100.0, 50.0, 200.0])

# Return tips
await lh.drop_tips(tiprack["A1:C1"])

C0TTid0001tt01tf1tl0871tv10650tg3tu0
C0TPid0002xp01629 01629 01629 00000&yp1458 1368 1278 0000&tm1 1 1 0&tt01tp2266tz2166th2450td0
C0ASid0003at0 0 0 0&tm1 1 1 0&xp04333 04333 04333 00000&yp1457 1367 1277 0000&th2450te2450lp2000 2000 2000 2000&ch000 000 000 000&zl1866 1866 1866 1866&po0100 0100 0100 0100&zu0032 0032 0032 0032&zr06180 06180 06180 06180&zx1866 1866 1866 1866&ip0000 0000 0000 0000&it0 0 0 0&fp0000 0000 0000 0000&av01083 00563 02110 01083&as2500 2500 2500 2500&ta000 000 000 000&ba0000 0000 0000 0000&oa000 000 000 000&lm0 0 0 0&ll1 1 1 1&lv1 1 1 1&zo000 000 000 000&ld00 00 00 00&de0020 0020 0020 0020&wt10 10 10 10&mv00000 00000 00000 00000&mc00 00 00 00&mp000 000 000 000&ms1000 1000 1000 1000&mh0000 0000 0000 0000&gi000 000 000 000&gj0gk0lk0 0 0 0&ik0000 0000 0000 0000&sd0500 0500 0500 0500&se0500 0500 0500 0500&sz0300 0300 0300 0300&io0000 0000 0000 0000&
C0DSid0004dm2 2 2 2&tm1 1 1 0&xp04333 04333 04333 00000&yp1187 1097 1007 0000&zx1866 1866 1866 1866&lp2000 2000 2000 200

### Head96: 96-channel head

In [6]:
tiprack96 = deck.get_resource("tips_02")

await lh.pick_up_tips96(tiprack96)
await lh.aspirate96(plate, volume=50)
await lh.dispense96(plate, volume=50)
await lh.drop_tips96(tiprack96)

H0DQid0006dq11281dv13500du00000dr900000dw15
C0EPid0007xs01629xd0yh2418tt01wu0za2166zh2450ze2450
C0EAid0008aa0xs04333xd0yh1457zh2450ze2450lz1999zt1866pp0100zm1866zv0032zq06180iw000ix0fh000af00500ag2500vt050bv00000wv00050cm0cs1bs0020wh10hv00000hc00hp000mj000hs1200cwFFFFFFFFFFFFFFFFFFFFFFFFcr000cj0cx0
C0EDid0009da2xs04333xd0yh1457zm1866zv0032zq06180lz1999zt1866pp0100iw000ix0fh000zh2450ze2450df00500dg1200es0050ev000vt050bv00000cm0cs1ej00bs0020wh50hv00000hc00hp000mj000hs1200cwFFFFFFFFFFFFFFFFFFFFFFFFcr000cj0cx0
C0ERid0010xs01629xd0yh2418za2164zh2450ze2450


### iSWAP: plate gripper arm

In [7]:
# Move plate_01 from carrier slot 0 to slot 1 using iSWAP
await lh.move_resource(plate, plt_car[2], pickup_distance_from_top=13.2)

C0PPid0011xs04829xd0yj1142yd0zj1841zd0gr1th2800te2800gw4go1308gb1245gt20ga0gc0
C0PRid0012xs04829xd0yj3062yd0zj1841zd0th2800te2800gr1go1308ga0gc0


In [8]:
# Move it back
await lh.move_resource(plate, plt_car[0], pickup_distance_from_top=13.2)

C0PPid0013xs04829xd0yj3062yd0zj1841zd0gr1th2800te2800gw4go1308gb1245gt20ga0gc0
C0PRid0014xs04829xd0yj1142yd0zj1841zd0th2800te2800gr1go1308ga0gc0


### CoRe gripper: channel-mounted plate gripper

In [9]:
plate2 = deck.get_resource("plate_02")

# Move plate_02 using CoRe gripper (channels 7+8)
await lh.move_resource(
  plate2,
  plt_car[2],
  pickup_distance_from_top=13.2,
  use_arm="core",
  core_front_channel=7,
)

C0PGid0015th2800
C0ZPid0016xs04829xd0yj2102yv0050zj1841zy0500yo0885yg0825yw15th2800te2800
C0ZRid0017xs04829xd0yj3062zj1841zi000zy0500yo0885th2800te2800
C0ZSid0018xs13375xd0ya1250yb1070tp2150tz2050th2800te2800


In [10]:
# Move it back
await lh.move_resource(
  plate2,
  plt_car[1],
  pickup_distance_from_top=13.2,
  use_arm="core",
  core_front_channel=7,
)

C0ZTid0019xs13375xd0ya1250yb1070pa07pb08tp2350tz2250th2800tt14
C0ZPid0020xs04829xd0yj3062yv0050zj1841zy0500yo0885yg0825yw15th2800te2800
C0ZRid0021xs04829xd0yj2102zj1841zi000zy0500yo0885th2800te2800
C0ZSid0022xs13375xd0ya1250yb1070tp2150tz2050th2800te2800


### Cleanup

In [11]:
await lh.stop()

## Part 2: New API

Using `STAR` device + `STARChatterboxDriver` with the new capability architecture.

### Setup

In [ ]:
from pylabrobot.hamilton.liquid_handlers.star.star import STAR
from pylabrobot.resources import (
  PLT_CAR_L5AC_A00,
  TIP_CAR_480_A00,
  Cor_96_wellplate_360ul_Fb,
  hamilton_96_tiprack_1000uL_filter,
)
from pylabrobot.resources.hamilton import STARDeck

deck2 = STARDeck()
star = STAR(deck=deck2, chatterbox=True)

In [14]:
# Deck layout
tip_car2 = TIP_CAR_480_A00(name="tip_carrier_2")
tip_car2[0] = hamilton_96_tiprack_1000uL_filter(name="tips_2_01")
tip_car2[1] = hamilton_96_tiprack_1000uL_filter(name="tips_2_02")
deck2.assign_child_resource(tip_car2, rails=3)

plt_car2 = PLT_CAR_L5AC_A00(name="plate_carrier_2")
plt_car2[0] = Cor_96_wellplate_360ul_Fb(name="plate_2_01")
plt_car2[1] = Cor_96_wellplate_360ul_Fb(name="plate_2_02")
deck2.assign_child_resource(plt_car2, rails=15)

In [15]:
await star.setup()

print(f"pip: {star.pip.num_channels} channels")
print(f"head96: {star.head96 is not None}")
print(f"iswap: {star.iswap is not None}")

pip: 8 channels
head96: True
iswap: True


In [24]:
star._capabilities

### PIP: independent channel pipetting

TODO: implement `STARPIPBackend.pick_up_tips`, `aspirate`, `dispense`, `drop_tips`

In [ ]:
tiprack = deck2.get_resource("tips_2_01")
plate = deck2.get_resource("plate_2_01")

await star.pip.pick_up_tips(tiprack["A1:C1"])
await star.pip.aspirate(plate["A1:C1"], vols=[100.0, 50.0, 200.0])
await star.pip.dispense(plate["D1:F1"], vols=[100.0, 50.0, 200.0])
await star.pip.drop_tips(tiprack["A1:C1"])

C0TTid0001tt01tf1tl0871tv10650tg3tu0
C0TPid0002xp01629 01629 01629 00000&yp1458 1368 1278 0000&tm1 1 1 0&tt01tp2266tz2166th2450td0
C0ASid0003at0 0 0 0&tm1 1 1 0&xp04333 04333 04333 00000&yp1457 1367 1277 0000&th2450te2450lp2000 2000 2000 2000&ch000 000 000 000&zl1866 1866 1866 1866&po0100 0100 0100 0100&zu0032 0032 0032 0032&zr06180 06180 06180 06180&zx1866 1866 1866 1866&ip0000 0000 0000 0000&it0 0 0 0&fp0000 0000 0000 0000&av01000 00500 02000 01000&as1000 1000 1000 1000&ta000 000 000 000&ba0000 0000 0000 0000&oa000 000 000 000&lm0 0 0 0&ll1 1 1 1&lv1 1 1 1&zo000 000 000 000&ld00 00 00 00&de1000 1000 1000 1000&wt00 00 00 00&mv00000 00000 00000 00000&mc00 00 00 00&mp000 000 000 000&ms1000 1000 1000 1000&mh0000 0000 0000 0000&gi000 000 000 000&gj0gk0lk0 0 0 0&ik0000 0000 0000 0000&sd0500 0500 0500 0500&se0500 0500 0500 0500&sz0300 0300 0300 0300&io0000 0000 0000 0000&
C0DSid0004dm2 2 2 2&tm1 1 1 0&xp04333 04333 04333 00000&yp1187 1097 1007 0000&zx1866 1866 1866 1866&lp2000 2000 2000 200

### Head96: 96-channel head

TODO: implement `STARHead96Backend.pick_up_tips96`, `aspirate96`, `dispense96`, `drop_tips96`

In [17]:
star.head96.backend is star._driver.head96

True

In [ ]:
tiprack96 = deck2.get_resource("tips_2_02")

await star.head96.pick_up_tips(tiprack96)
await star.head96.aspirate(plate, volume=50)
await star.head96.dispense(plate, volume=50)
await star.head96.drop_tips(tiprack96)

### iSWAP: plate gripper arm

The iSWAP backend is already implemented. The `OrientableArm` frontend provides `pick_up_resource` / `drop_resource` / `move_resource`.

In [19]:
star.iswap.backend is star._driver.iswap

True

In [20]:
plate2 = deck2.get_resource("plate_2_01")

# Move plate between carrier slots using iSWAP
await star.iswap.move_resource(plate2, plt_car2[2], pickup_distance_from_top=13.2)

C0PPid0011xs04829xd0yj1142yd0zj1841zd0gr1th2800te2800gw4go1308gb1245gt20ga0gc0
C0PRid0012xs04829xd0yj3062yd0zj1841zd0th2800te2800gr1go1308ga0gc0


Moving using a function that takes any `OrientableArm`:

In [ ]:
from pylabrobot.arms.orientable_arm import OrientableArm


async def move_resource(resource, destination, arm: OrientableArm):
  await arm.move_resource(resource, destination, pickup_distance_from_top=13.2)


await move_resource(plate2, plt_car2[0], arm=star.iswap)
# await move_resource(plate2, plt_car2[0], arm=pf400.arm)

C0PPid0013xs04829xd0yj3062yd0zj1841zd0gr1th2800te2800gw4go1308gb1245gt20ga0gc0
C0PRid0014xs04829xd0yj1142yd0zj1841zd0th2800te2800gr1go1308ga0gc0


### CoRe gripper

In [ ]:
async with star.core_grippers(front_channel=7) as arm:
  await move_resource(plate2, plt_car2[2], arm=arm)

C0PGid0015th2800
C0ZTid0016xs13375xd0ya1250yb1070pa07pb08tp2350tz2250th2800tt14
arm type: <class 'pylabrobot.arms.arm.GripperArm'>
C0ZPid0017xs04829xd0yj1142yv0050zj1841zy0500yo0885yg0825yw15th2800te2800
C0ZRid0018xs04829xd0yj3062zj1841zi000zy0500yo0885th2800te2800
C0ZSid0019xs13375xd0ya1250yb1070tp2150tz2050th2800te2800


### Cleanup

In [23]:
# await star.stop()